# Load a CSV file into MySQL with Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kartechbabu/MySQL_notebooks/blob/main/csv_to_mysql_python.ipynb)

**ITOM 6265 · Database Design for Business.** Explained in the book chapter
*Loading Data from Files* (Part III). Checked November 17, 2026 with
pandas 3.0, SQLAlchemy 2.1, PyMySQL 1.2 and MySQL 8.0.45.

The plan is three lines of work: **read** the CSV file into a pandas
DataFrame, **connect** to your MySQL database, and **append** the rows to a
table with `DataFrame.to_sql`.

**Your database account.** Never type your password into a notebook.

* **In Colab:** click the key icon (Secrets) in the left sidebar and add
  `DB_HOST`, `DB_PORT`, `DB_USER`, `DB_PASSWORD` and `DB_NAME`, each with
  *Notebook access* switched on.
* **On your computer:** put the same five lines in a `.env` file next to the
  notebook (`DB_HOST=...`).

If neither is found, the notebook asks you, and hides what you type for the
password.

## 1. Install and connect

In [1]:
# Colab already has pandas and SQLAlchemy; add the MySQL driver and dotenv.
import subprocess, sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymysql", "python-dotenv"], check=True)

In [2]:
import os
from getpass import getpass

from dotenv import load_dotenv

# On your computer: read DB_HOST, DB_USER, ... from a .env file, if there is one.
load_dotenv()


def setting(name):
    """Find a connection setting: .env first, then Colab Secrets, then ask."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata  # exists only in Colab
        return userdata.get(name)
    except Exception:
        pass
    return getpass(f"{name}: ") if name == "DB_PASSWORD" else input(f"{name}: ")

In [3]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Describe the connection: driver, account, server, database.
url = URL.create(
    "mysql+pymysql",                  # MySQL, through the PyMySQL driver
    username=setting("DB_USER"),
    password=setting("DB_PASSWORD"),  # typed as is: no escaping needed for @, # or /
    host=setting("DB_HOST"),
    port=int(setting("DB_PORT")),
    database=setting("DB_NAME"),
)
# The engine opens connections when needed. PyMySQL encrypts them when the
# server supports it, as the class server does.
engine = create_engine(url)



def why(err):
    """The server's own error message, without the long wrapper text."""
    while getattr(err, "orig", None) is not None or err.__cause__ is not None:
        err = getattr(err, "orig", None) or err.__cause__
    return err


# Test the connection, and check that it is encrypted (a cipher name, not empty).
with engine.connect() as conn:
    print(conn.execute(text("SELECT DATABASE(), VERSION()")).one())
    print(conn.execute(text("SHOW SESSION STATUS LIKE 'Ssl_cipher'")).one())

('0000_dem', '8.0.45')
('Ssl_cipher', 'TLS_AES_256_GCM_SHA384')


## 2. Read the CSV file into a DataFrame

`read_csv` accepts a web address as well as a file name, so the notebook reads
the sample file straight from the course's GitHub repository. Addresses with
commas inside are wrapped in double quotes in the file; `read_csv` handles
that by default.

In [4]:
import pandas as pd

# The sample files live in the notebook's GitHub repository.
DATA = "https://raw.githubusercontent.com/kartechbabu/MySQL_notebooks/main/data/"

df = pd.read_csv(DATA + "london_restaurants.csv")  # one row per line of the file
print(df.shape)                                    # (rows, columns)
df.head()

(15, 9)


,name,city,address,longitude,latitude,average_cost,price_range,aggregate_rating,votes
0,Gymkhana,London,"42 Albemarle Street, Mayfair, London W1S 4JH",-0.141645,51.508515,80,4,4.7,214
1,Bocca Di Lupo,London,"12 Archer Street, Soho, London W1D 7BB",-0.133947,51.511629,45,3,4.5,571
2,Flat Iron,London,"17 Beak Street, Soho, London W1F 9RW",-0.138343,51.512069,35,2,4.9,309
3,Duck & Waffle,London,"Heron Tower, 110 Bishopsgate, City of London, ...",-0.080963,51.516284,55,3,4.9,706
4,Dishoom,London,"7 Boundary Street, Shoreditch, London E2 7JE",-0.076581,51.524536,55,3,4.5,305


## 3. Case 1: append to a table that already exists

This is the everyday case: the table was designed in advance, and new data
arrives as a file. Here we create the table first, as its designer would:
real types, a **primary key**, and a `UNIQUE` rule so the same restaurant
cannot be loaded twice. (Run this cell only once; it replaces the table.)

In [5]:
create_sql = """
CREATE TABLE london_restaurants (
  restaurant_id     INT           NOT NULL AUTO_INCREMENT PRIMARY KEY,
  name              VARCHAR(100)  NOT NULL,
  city              VARCHAR(50)   NOT NULL,
  address           VARCHAR(255)  NOT NULL,
  longitude         DECIMAL(9,6),
  latitude          DECIMAL(9,6),
  average_cost      DECIMAL(7,2),
  price_range       TINYINT,
  aggregate_rating  DECIMAL(2,1),
  votes             INT,
  CONSTRAINT london_restaurants_uq UNIQUE (name, address)
)"""
# engine.begin() runs the statements and commits them together.
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS london_restaurants"))
    conn.execute(text(create_sql))
print("table created")

table created


Now the load itself is one line. `if_exists="append"` adds the rows to the
existing table; `index=False` leaves out the DataFrame's row numbers, which
are not a column of the table. The DataFrame's column names must match the
table's.

In [6]:
rows = df.to_sql("london_restaurants", engine, index=False, if_exists="append")
print(rows, "rows loaded")

15 rows loaded


In [7]:
# Check: the table should hold as many rows as the file.
pd.read_sql("SELECT restaurant_id, name, average_cost, votes FROM london_restaurants "
            "ORDER BY restaurant_id LIMIT 5", engine)

,restaurant_id,name,average_cost,votes
0,1,Gymkhana,80.0,214
1,2,Bocca Di Lupo,45.0,571
2,3,Flat Iron,35.0,309
3,4,Duck & Waffle,55.0,706
4,5,Dishoom,55.0,305


## 4. Case 2: a new table

**The recommended way** is the one above: write the `CREATE TABLE` yourself,
then append. You decide the types and the keys.

**The quick way** lets pandas create the table from the DataFrame. On the
class server this fails, because every table must have a primary key and
pandas does not create one:

In [8]:
try:
    df.to_sql("london_restaurants_quick", engine, index=False)
except Exception as err:
    print("Refused:", why(err))

Refused: (3750, "Unable to create or change a table without a primary key, when the system variable 'sql_require_primary_key' is set. Add a primary key to the table or unset this variable to avoid this message. Note that tables without a primary key can cause performance problems in row-based replication, so please consult your DBA before changing this setting.")


You *can* switch the primary-key rule off for one connection, let pandas
create the table, and add a key afterwards. Look at the types pandas chose:
`TEXT` with no length limit, `DOUBLE` for coordinates, `BIGINT` for a price.
That is why the recommended way is better.

In [9]:
with engine.begin() as conn:
    conn.execute(text("SET SESSION sql_require_primary_key = 0"))  # this connection only
    df.to_sql("london_restaurants_quick", conn, index=False)
    conn.execute(text("ALTER TABLE london_restaurants_quick "
                      "ADD restaurant_id INT NOT NULL AUTO_INCREMENT PRIMARY KEY FIRST"))
    conn.execute(text("SET SESSION sql_require_primary_key = 1"))

pd.read_sql("SELECT column_name, column_type FROM information_schema.columns "
            "WHERE table_schema = DATABASE() AND table_name = 'london_restaurants_quick' "
            "ORDER BY ordinal_position", engine)

,COLUMN_NAME,COLUMN_TYPE
0,restaurant_id,int
1,name,text
2,city,text
3,address,text
4,longitude,double
5,latitude,double
6,average_cost,bigint
7,price_range,bigint
8,aggregate_rating,double
9,votes,bigint


**Never use `if_exists="replace"` on a table you care about.** pandas first
**drops** the existing table, then tries to create a new one, which the class
server refuses. The result is no table at all:

In [10]:
try:
    df.to_sql("london_restaurants_quick", engine, index=False, if_exists="replace")
except Exception as err:
    print("Refused:", why(err))

pd.read_sql("SELECT COUNT(*) AS tables_named_quick FROM information_schema.tables "
            "WHERE table_schema = DATABASE() AND table_name = 'london_restaurants_quick'", engine)

Refused: (3750, "Unable to create or change a table without a primary key, when the system variable 'sql_require_primary_key' is set. Add a primary key to the table or unset this variable to avoid this message. Note that tables without a primary key can cause performance problems in row-based replication, so please consult your DBA before changing this setting.")


,tables_named_quick
0,0


## 5. A newer file: append, or update what is already there

`london_restaurants_update.csv` has five new restaurants and a newer row for
**Gymkhana**, which is already loaded. (Its vote count, 230, is invented to
show an update.) Appending fails on the duplicate, and because one `INSERT`
is all-or-nothing, **none** of the six rows is added:

In [11]:
df2 = pd.read_csv(DATA + "london_restaurants_update.csv")
try:
    df2.to_sql("london_restaurants", engine, index=False, if_exists="append")
except Exception as err:
    print("Refused:", why(err))

pd.read_sql("SELECT COUNT(*) AS rows_in_table FROM london_restaurants", engine)

Refused: (1062, "Duplicate entry 'Gymkhana-42 Albemarle Street, Mayfair, London W1S 4JH' for key 'london_restaurants.london_restaurants_uq'")


,rows_in_table
0,15


To insert the new rows **and** update the existing one, use an *upsert*:
`INSERT … ON DUPLICATE KEY UPDATE`. pandas has no option for it, so write the
SQL and pass the DataFrame's rows to it; `:name` placeholders are filled from
each row, and `AS new` names the incoming row.

In [12]:
upsert_sql = text("""
INSERT INTO london_restaurants
  (name, city, address, longitude, latitude,
   average_cost, price_range, aggregate_rating, votes)
VALUES
  (:name, :city, :address, :longitude, :latitude,
   :average_cost, :price_range, :aggregate_rating, :votes) AS new
ON DUPLICATE KEY UPDATE votes = new.votes, aggregate_rating = new.aggregate_rating
""")
with engine.begin() as conn:
    conn.execute(upsert_sql, df2.to_dict("records"))  # one dict per row

pd.read_sql("SELECT name, votes FROM london_restaurants "
            "WHERE name IN ('Gymkhana', 'Bone Daddies')", engine)

,name,votes
0,Bone Daddies,418
1,Gymkhana,230


In [13]:
pd.read_sql("SELECT COUNT(*) AS rows_in_table FROM london_restaurants", engine)

,rows_in_table
0,20


## 6. A messy file: load as text, then convert in SQL

Real exports have stray spaces, empty cells and odd values. Load every column
**as text** into a staging table, so nothing is refused, then convert with one
`INSERT … SELECT`: `TRIM` removes spaces, `NULLIF(…, '')` turns empty cells
into `NULL`, and `CAST` sets the type.

In [14]:
raw = pd.read_csv(DATA + "london_restaurants.csv", dtype=str)  # every value as text

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS london_restaurants_staging"))
    conn.execute(text("""
        CREATE TABLE london_restaurants_staging (
          row_id INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
          name VARCHAR(255), city VARCHAR(255), address VARCHAR(255),
          longitude VARCHAR(255), latitude VARCHAR(255), average_cost VARCHAR(255),
          price_range VARCHAR(255), aggregate_rating VARCHAR(255), votes VARCHAR(255))"""))

print(raw.to_sql("london_restaurants_staging", engine, index=False, if_exists="append"), "rows staged")

15 rows staged


In [15]:
convert_sql = text("""
INSERT INTO london_restaurants_clean
  (name, city, address, longitude, latitude,
   average_cost, price_range, aggregate_rating, votes)
SELECT TRIM(name), TRIM(city), TRIM(address),
       CAST(NULLIF(TRIM(longitude), '') AS DECIMAL(9,6)),
       CAST(NULLIF(TRIM(latitude), '') AS DECIMAL(9,6)),
       CAST(NULLIF(TRIM(average_cost), '') AS DECIMAL(7,2)),
       CAST(NULLIF(TRIM(price_range), '') AS UNSIGNED),
       CAST(NULLIF(TRIM(aggregate_rating), '') AS DECIMAL(2,1)),
       CAST(NULLIF(TRIM(votes), '') AS UNSIGNED)
FROM london_restaurants_staging
ORDER BY row_id""")

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS london_restaurants_clean"))
    conn.execute(text("CREATE TABLE london_restaurants_clean LIKE london_restaurants"))  # typed target
    print(conn.execute(convert_sql).rowcount, "rows converted")

pd.read_sql("SELECT name, average_cost, aggregate_rating, votes FROM london_restaurants_clean "
            "ORDER BY restaurant_id LIMIT 3", engine)

15 rows converted


,name,average_cost,aggregate_rating,votes
0,Gymkhana,80.0,4.7,214
1,Bocca Di Lupo,45.0,4.5,571
2,Flat Iron,35.0,4.9,309


## 7. Clean up

Drop the practice tables (set `CLEAN_UP = False` to keep them and explore).

In [16]:
CLEAN_UP = True
if CLEAN_UP:
    with engine.begin() as conn:
        conn.execute(text("DROP TABLE IF EXISTS london_restaurants_staging, "
                          "london_restaurants_quick, london_restaurants_clean, london_restaurants"))
    print("practice tables dropped")
engine.dispose()

practice tables dropped


## What to remember

* `pd.read_csv` reads the file; `df.to_sql(..., if_exists="append", index=False)`
  appends it to an existing table.
* Create new tables yourself (types, primary key, `UNIQUE` rules), then
  append. Never `if_exists="replace"` on a table you need.
* Keep passwords out of notebooks: Colab Secrets or a `.env` file.
* For a newer file, upsert with `INSERT … ON DUPLICATE KEY UPDATE`; for a
  messy file, stage as text and convert with `INSERT … SELECT`.